In [1]:
# Suitability check for the Singapore geotagged-tweets dataset (reads selected columns only)
from huggingface_hub import snapshot_download
import pandas as pd, glob

repo = "cestwc/SG-subzone-poi-sentiment-estate"
path = snapshot_download(repo_id=repo, repo_type="dataset", allow_patterns=["data/*.parquet"])
cols = ["id", "text", "lang", "source", "latitude", "longitude", "subzone",
        "planning_area", "poi_id", "year_created_at"]
sg = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted(glob.glob(f"{path}/data/*.parquet"))],
               ignore_index=True)

print(len(sg), "rows;", sg[["latitude", "longitude"]].drop_duplicates().shape[0], "distinct coordinate pairs")
print(sg["year_created_at"].value_counts().sort_index())
print(sg["lang"].value_counts().head(8))
print(sg["source"].str.extract(r">([^<]+)<")[0].value_counts().head(10))   # app noise
print("share with POI:", sg["poi_id"].notna().mean())
print(sg["subzone"].value_counts().describe())                              # posts per subzone
print(sg["text"].str.lower().value_counts().head(10))                       # templated/bot texts

/home/roman/Documents/university/thesis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:26<00:00,  5.25s/it]


1025135 rows; 109158 distinct coordinate pairs
year_created_at
2018    403516
2019    307709
2020    182592
2021    131240
2022        78
Name: count, dtype: int64
lang
en     778939
in      49029
tl      46185
ja      39713
und     29684
zh      18278
es       9155
et       8390
Name: count, dtype: int64
0
Instagram               612124
Foursquare              219821
Twitter for iPhone       64012
Twitter for Android      48232
OLIO Articles Plugin     32357
Foursquare Swarm         11781
Trendsmap Alerting        5539
Tweetbot for iΟS          5477
Flamingo for Android      4724
World Cities              4447
Name: count, dtype: int64
share with POI: 0.05328371385232189
count       320.000000
mean       3196.665625
std       10406.577885
min           1.000000
25%         401.750000
50%        1063.000000
75%        2759.250000
max      162306.000000
Name: count, dtype: float64
text
singapore: 7:16pm: sunset                               41
singapore: 7:06pm: sunset                  

In [2]:
# Clean-subset diagnostics: which posts are usable for the spatial analysis
import re
import h3

# 1) Concentration of coordinates: how much do the top places take?
pair_counts = sg.groupby(["latitude", "longitude"]).size().sort_values(ascending=False)
print("top-10 coordinate pairs share:", pair_counts.head(10).sum() / len(sg))
print(pair_counts.head(10))

# 2) Flags (do not delete, only flag)
src = sg["source"].str.extract(r">([^<]+)<")[0].fillna("")
sg["src"] = src
sg["is_checkin"] = src.str.contains("Foursquare|Swarm", case=False)
sg["is_bot"] = src.str.contains("Trendsmap|World Cities|OLIO|Alerting", case=False)
sg["is_ig_template"] = (src.eq("Instagram")
                        & sg["text"].str.contains(r"(?i)^(just posted a photo|vient de publier)", regex=True))
sg["text_norm"] = sg["text"].str.lower().str.replace(r"https?://\S+", "", regex=True).str.strip()

# 3) Usable candidate pool: not check-in, not bot, not templated Instagram, has real text
usable = sg[~sg["is_checkin"] & ~sg["is_bot"] & ~sg["is_ig_template"] & (sg["text_norm"].str.len() >= 15)]
print("usable posts:", len(usable), f"({len(usable)/len(sg):.1%})")
print(usable["src"].value_counts().head(8))
print("distinct coords in usable:", usable[["latitude", "longitude"]].drop_duplicates().shape[0])
print("top-10 coord share in usable:",
      usable.groupby(["latitude", "longitude"]).size().sort_values(ascending=False).head(10).sum() / len(usable))

# 4) Density on H3 grid
for r in (8, 9):
    cells = [h3.latlng_to_cell(a, b, r) for a, b in zip(usable["latitude"], usable["longitude"])]
    vc = pd.Series(cells).value_counts()
    print(f"res {r}: cells={len(vc)}, median={vc.median():.0f}, share cells with 10+ posts={(vc >= 10).mean():.1%}")

# 5) The giant subzone: what is it and how concentrated?
top_sz = sg["subzone"].value_counts().index[0]
big = sg[sg["subzone"] == top_sz]
print(top_sz, len(big), "distinct coords:", big[["latitude", "longitude"]].drop_duplicates().shape[0])
print(big.groupby(["latitude", "longitude"]).size().sort_values(ascending=False).head(5))

top-10 coordinate pairs share: 0.20033459007837992
latitude  longitude 
1.293060  103.856003    138593
1.360682  103.815697     29823
1.283887  103.859573      6058
1.378400  103.764603      5817
1.289410  103.849998      5744
1.370000  103.830002      4448
1.353767  103.987846      4179
1.281517  103.865753      4118
1.355379  103.867744      3301
1.300960  103.838959      3289
dtype: int64


/tmp/ipykernel_13660/4152758176.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & sg["text"].str.contains(r"(?i)^(just posted a photo|vient de publier)", regex=True))


usable posts: 673534 (65.7%)
src
Instagram               554952
Twitter for iPhone       52801
Twitter for Android      40224
Tweetbot for iΟS          4977
Flamingo for Android      4463
dlvr.it                   3061
Path                      2533
Squarespace               1756
Name: count, dtype: int64
distinct coords in usable: 72849
top-10 coord share in usable: 0.26917720560506225
res 8: cells=729, median=122, share cells with 10+ posts=80.4%
res 9: cells=3427, median=21, share cells with 10+ posts=63.9%
MARINA CENTRE 162306 distinct coords: 2233
latitude  longitude 
1.29306   103.856003    138593
1.28986   103.855919      1438
1.29216   103.858994      1407
1.29047   103.860046       662
1.29122   103.857803       593
dtype: int64


In [4]:
# Flag coarse-location posts and venue hubs; recompute density on the clean part
cols = ["id", "latitude", "longitude", "place_place_type", "place_name"]
pl = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted(glob.glob(f"{path}/data/*.parquet"))],
               ignore_index=True)
sg["place_place_type"] = pl["place_place_type"].values
sg["place_name"] = pl["place_name"].values

# 1) What kind of places do the big coordinates belong to?
for (la, lo) in [(1.29306, 103.856003), (1.360682, 103.815697), (1.353767, 103.987846)]:
    m = (sg["latitude"].round(6) == la) & (sg["longitude"].round(6) == lo)
    print((la, lo), m.sum(), sg.loc[m, "place_place_type"].value_counts().head(3).to_dict(),
          sg.loc[m, "place_name"].value_counts().head(3).to_dict())

# 2) Place-type distribution overall (poi / neighborhood / city / admin ...)
print(sg["place_place_type"].value_counts())

# 3) Flags: coarse = non-poi place type OR one of the two giant centroids
pair_n = sg.groupby(["latitude", "longitude"])["id"].transform("size")
sg["is_coarse_loc"] = (pair_n >= 20000)                      # tune after step 1/2
sg["is_hub"] = (pair_n >= 2000) & ~sg["is_coarse_loc"]       # real venues, cap later
print(sg[["is_coarse_loc", "is_hub"]].sum())

# 4) Density after removing coarse points
clean = usable[~sg.loc[usable.index, "is_coarse_loc"]]
print("clean posts:", len(clean), "distinct coords:", clean[["latitude", "longitude"]].drop_duplicates().shape[0])
for r in (8, 9):
    cells = pd.Series([h3.latlng_to_cell(a, b, r) for a, b in zip(clean["latitude"], clean["longitude"])]).value_counts()
    print(f"res {r}: cells={len(cells)}, median={cells.median():.0f}, share 10+={(cells >= 10).mean():.1%}")

(1.29306, 103.856003) 138593 {'admin': 138593} {'Central Region': 138593}
(1.360682, 103.815697) 29823 {'admin': 29823} {'North Region': 29823}
(1.353767, 103.987846) 4193 {'poi': 2156, 'admin': 2037} {'Singapore Changi Airport (SIN)': 2156, 'East Region': 2037}
place_place_type
admin           905168
poi             110929
country           9036
neighborhood         2
Name: count, dtype: int64
is_coarse_loc    168416
is_hub            58796
dtype: int64
clean posts: 521296 distinct coords: 72847
res 8: cells=729, median=122, share 10+=80.4%
res 9: cells=3427, median=21, share 10+=63.9%


In [5]:
# Find remaining coarse points: modal coordinate per admin/country place, and "round" coordinates
adm = sg[sg["place_place_type"].isin(["admin", "country"])]
top_per_place = (adm.groupby(["place_name", "latitude", "longitude"]).size()
                    .reset_index(name="n")
                    .sort_values("n", ascending=False)
                    .groupby("place_name").head(2))
print(top_per_place.sort_values(["place_name", "n"], ascending=[True, False]).head(20))

# Share of each place's posts that sit in its single most common coordinate
share = (adm.groupby("place_name")
            .apply(lambda g: g.groupby(["latitude", "longitude"]).size().max() / len(g)))
print(share.sort_values(ascending=False).head(10))

# "Round" coordinates (<= 3 decimals) are usually centroids/approximations
dec3 = (sg["latitude"].mul(1000).round(3) % 1 == 0) & (sg["longitude"].mul(1000).round(3) % 1 == 0)
print("round-coordinate posts:", dec3.sum())
print(sg.loc[dec3].groupby(["latitude", "longitude"]).size().sort_values(ascending=False).head(10))

              place_name  latitude   longitude       n
19411     Central Region  1.293060  103.856003  138593
11134     Central Region  1.283887  103.859573    6058
67236        East Region  1.356755  103.989250    2419
68476        East Region  1.361360  103.990334    2215
71060       North Region  1.360682  103.815697   29823
76054       North Region  1.444500  103.776802    1616
80436  North-East Region  1.370000  103.830002    4448
78403  North-East Region  1.355379  103.867744    3301
85904          Singapore  1.286830  103.855003    3225
85926          Singapore  1.287022  103.854691     425
94199        West Region  1.378400  103.764603    5817
94267        West Region  1.378676  103.746292    1355
place_name
North Region         0.381052
Singapore            0.356906
Central Region       0.233683
West Region          0.085866
North-East Region    0.082609
East Region          0.021553
dtype: float64
round-coordinate posts: 12260
latitude  longitude 
1.370     103.830002    4448